# Module 3: Constraining the Model

In the previous module, we saw OLS memorize the training data (R²=1.0) and fail spectacularly on test data. The model had too much freedom — it could assign any coefficient to any feature, no matter how absurd.

**Regularization** adds a penalty to prevent this. Instead of minimizing just the prediction error, we also penalize large coefficients. This forces the model to be more conservative and focus on features that truly matter.

We'll explore three regularization methods:
- **Ridge (L2)** — shrinks all coefficients toward zero, but never exactly to zero
- **Lasso (L1)** — can push coefficients all the way to zero, performing feature selection
- **ElasticNet** — a blend of both Ridge and Lasso

The key hyperparameter is **alpha** (α) — it controls how strongly we penalize. Too low = overfit. Too high = underfit.

In [ ]:
#@title 🔧 Setup { display-mode: "form" }

import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

from helpers import load_dataset, ROLE_COLORS, apply_dark_theme, takeaway_box, metric_card, role_bar_colors
from helpers.styling import role_legend_html, styled_button, DARK_BG, AXES_BG, TEXT_COLOR, GREEN, RED, GRAY, MUTED_TEXT

# Load data
X, y, codebook, role_map = load_dataset()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape[0]} countries, {X_train_scaled.shape[1]} features")
print(f"Test set: {X_test_scaled.shape[0]} countries")

## Widget 1: Train a Regularized Model

Pick a regularization method and adjust the alpha penalty. Watch how the metrics change — especially the gap between train and test performance.

In [ ]:
#@title 🔧 Alpha Slider Widget { display-mode: "form" }

class AlphaSliderWidget:
    """Interactive widget for training regularized models with adjustable alpha."""

    def __init__(self):
        self.model_toggle = widgets.ToggleButtons(
            options=['Ridge', 'Lasso', 'ElasticNet'],
            value='Ridge',
            description='Model:',
            style={'button_width': '120px', 'description_width': '60px'}
        )

        self.alpha_slider = widgets.FloatLogSlider(
            value=100,
            base=10,
            min=-2,
            max=4,
            step=0.2,
            description='Alpha (α):',
            style={'description_width': '80px'},
            layout=widgets.Layout(width='500px'),
            readout_format='.2f'
        )

        self.train_btn = styled_button('Train Model', width='160px')
        self.train_btn.on_click(self._on_train)

        self.output = widgets.Output()

        # Store last trained model for other widgets
        self.last_model = None
        self.last_model_type = None
        self.last_alpha = None

    def _get_model(self, model_type, alpha):
        if model_type == 'Ridge':
            return Ridge(alpha=alpha)
        elif model_type == 'Lasso':
            return Lasso(alpha=alpha, max_iter=5000)
        else:
            return ElasticNet(alpha=alpha, max_iter=5000)

    def _on_train(self, _):
        model_type = self.model_toggle.value
        alpha = self.alpha_slider.value

        model = self._get_model(model_type, alpha)
        model.fit(X_train_scaled, y_train)

        train_r2 = model.score(X_train_scaled, y_train)
        test_r2 = model.score(X_test_scaled, y_test)
        overfit_gap = train_r2 - test_r2
        n_nonzero = int(np.sum(np.abs(model.coef_) > 1e-6))

        self.last_model = model
        self.last_model_type = model_type
        self.last_alpha = alpha

        # Color logic
        train_color = GREEN if train_r2 > 0.5 else RED
        test_color = GREEN if test_r2 > 0.5 else (GRAY if test_r2 > 0 else RED)
        gap_color = RED if overfit_gap > 0.3 else GREEN
        nonzero_color = MUTED_TEXT

        with self.output:
            clear_output(wait=True)
            title = widgets.HTML(
                f"<h3 style='color:{TEXT_COLOR}; margin-bottom:5px;'>"
                f"{model_type} (α = {alpha:.2f})</h3>"
            )
            cards = widgets.HBox([
                metric_card('Train R²', f'{train_r2:.3f}', color=train_color),
                metric_card('Test R²', f'{test_r2:.3f}', color=test_color),
                metric_card('Overfit Gap', f'{overfit_gap:.3f}', color=gap_color),
                metric_card('Non-zero Coefs', f'{n_nonzero}', color=nonzero_color),
            ])
            display(widgets.VBox([title, cards]))

    def create_ui(self):
        return widgets.VBox([
            self.model_toggle,
            self.alpha_slider,
            self.train_btn,
            self.output
        ])

alpha_widget = AlphaSliderWidget()

In [ ]:
#@title ▶️ Launch Alpha Slider { display-mode: "form" }
display(alpha_widget.create_ui())

## Widget 2: Alpha Sweep

How do we find the best alpha? Let's sweep across a wide range and plot the results. Watch the sweet spot where test R² peaks — that's the bias-variance tradeoff in action.

In [ ]:
#@title 🔧 Alpha Sweep Widget { display-mode: "form" }

class AlphaSweepWidget:
    """Sweep across alpha values and plot train/test R² and sparsity."""

    def __init__(self, alpha_widget_ref):
        self.alpha_widget_ref = alpha_widget_ref
        self.sweep_btn = styled_button('Run Alpha Sweep', width='180px')
        self.sweep_btn.on_click(self._on_sweep)
        self.output = widgets.Output()

    def _on_sweep(self, _):
        model_type = self.alpha_widget_ref.model_toggle.value
        alphas = np.logspace(-2, 4, 30)

        train_scores = []
        test_scores = []
        n_nonzeros = []

        for alpha in alphas:
            if model_type == 'Ridge':
                model = Ridge(alpha=alpha)
            elif model_type == 'Lasso':
                model = Lasso(alpha=alpha, max_iter=5000)
            else:
                model = ElasticNet(alpha=alpha, max_iter=5000)

            model.fit(X_train_scaled, y_train)
            train_scores.append(model.score(X_train_scaled, y_train))
            test_scores.append(model.score(X_test_scaled, y_test))
            n_nonzeros.append(int(np.sum(np.abs(model.coef_) > 1e-6)))

        best_idx = np.argmax(test_scores)
        best_alpha = alphas[best_idx]
        best_test_r2 = test_scores[best_idx]

        with self.output:
            clear_output(wait=True)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            apply_dark_theme(fig, [ax1, ax2])

            # Left: Train/Test R² vs alpha
            ax1.plot(alphas, train_scores, color=GREEN, linewidth=2, label='Train R²')
            ax1.plot(alphas, test_scores, color=RED, linewidth=2, label='Test R²')
            ax1.axvline(best_alpha, color=MUTED_TEXT, linestyle='--', alpha=0.7,
                        label=f'Best α = {best_alpha:.2f}')
            ax1.set_xscale('log')
            ax1.set_xlabel('Alpha (α)')
            ax1.set_ylabel('R² Score')
            ax1.set_title(f'{model_type}: R² vs Alpha')
            ax1.legend(facecolor=AXES_BG, edgecolor='#4a4a6a', labelcolor=TEXT_COLOR)
            ax1.annotate(f'Best Test R² = {best_test_r2:.3f}',
                         xy=(best_alpha, best_test_r2),
                         xytext=(best_alpha * 5, best_test_r2 - 0.15),
                         arrowprops=dict(arrowstyle='->', color=MUTED_TEXT),
                         color=TEXT_COLOR, fontsize=10)

            # Right: Non-zero coefficients vs alpha
            ax2.plot(alphas, n_nonzeros, color='#a78bfa', linewidth=2)
            ax2.set_xscale('log')
            ax2.set_xlabel('Alpha (α)')
            ax2.set_ylabel('Non-zero Coefficients')
            ax2.set_title(f'{model_type}: Feature Sparsity vs Alpha')

            if model_type == 'Ridge':
                ax2.text(0.5, 0.5, 'Ridge does not zero out features.\nAll coefficients remain non-zero.',
                         transform=ax2.transAxes, ha='center', va='center',
                         color=MUTED_TEXT, fontsize=12, style='italic')

            plt.tight_layout()
            plt.show()

    def create_ui(self):
        return widgets.VBox([
            self.sweep_btn,
            self.output
        ])

sweep_widget = AlphaSweepWidget(alpha_widget)

In [ ]:
#@title ▶️ Launch Alpha Sweep { display-mode: "form" }
display(sweep_widget.create_ui())

## Widget 3: Coefficient Bar Chart

Which features does the model rely on most? The bar chart below shows the top-20 features by coefficient magnitude, color-coded by their role (causal, bizarre, or incidental). Train a model above first, then click below to inspect its coefficients.

In [ ]:
#@title 🔧 Coefficient Bar Chart Widget { display-mode: "form" }

class CoefficientBarWidget:
    """Horizontal bar chart of top-20 features by coefficient magnitude."""

    def __init__(self, alpha_widget_ref):
        self.alpha_widget_ref = alpha_widget_ref
        self.show_btn = styled_button('Show Coefficients', width='200px')
        self.show_btn.on_click(self._on_show)
        self.output = widgets.Output()

    def _on_show(self, _):
        model_type = self.alpha_widget_ref.model_toggle.value
        alpha = self.alpha_widget_ref.alpha_slider.value

        # Train fresh model with current settings
        if model_type == 'Ridge':
            model = Ridge(alpha=alpha)
        elif model_type == 'Lasso':
            model = Lasso(alpha=alpha, max_iter=5000)
        else:
            model = ElasticNet(alpha=alpha, max_iter=5000)

        model.fit(X_train_scaled, y_train)

        coefs = pd.Series(model.coef_, index=X.columns)
        top20 = coefs.abs().nlargest(20)
        top20_coefs = coefs[top20.index]

        # Sort by actual value for display
        top20_coefs = top20_coefs.sort_values()

        features = top20_coefs.index.tolist()
        values = top20_coefs.values
        colors = role_bar_colors(features, role_map)

        with self.output:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(10, 8))
            apply_dark_theme(fig, ax)

            bars = ax.barh(range(len(features)), values, color=colors, edgecolor='none', height=0.7)
            ax.set_yticks(range(len(features)))
            ax.set_yticklabels(features, fontsize=9)
            ax.set_xlabel('Coefficient Value')
            ax.set_title(f'{model_type} (α={alpha:.2f}) — Top 20 Coefficients')
            ax.axvline(0, color=MUTED_TEXT, linewidth=0.8, alpha=0.5)

            # Annotate coefficient values
            for i, (v, bar) in enumerate(zip(values, bars)):
                offset = max(abs(values)) * 0.02
                ha = 'left' if v >= 0 else 'right'
                x_pos = v + offset if v >= 0 else v - offset
                ax.text(x_pos, i, f'{v:.0f}', va='center', ha=ha,
                        color=TEXT_COLOR, fontsize=8)

            plt.tight_layout()
            plt.show()

            display(role_legend_html())

    def create_ui(self):
        return widgets.VBox([
            self.show_btn,
            self.output
        ])

coef_widget = CoefficientBarWidget(alpha_widget)

In [ ]:
#@title ▶️ Launch Coefficient Chart { display-mode: "form" }
display(coef_widget.create_ui())

## Widget 4: Ridge vs Lasso Comparison

Ridge and Lasso take fundamentally different approaches. Ridge keeps all features but shrinks them. Lasso is more aggressive — it eliminates features entirely. Let's compare them head-to-head at the same alpha.

In [ ]:
#@title 🔧 Ridge vs Lasso Comparison Widget { display-mode: "form" }

class RidgeLassoComparisonWidget:
    """Side-by-side comparison of Ridge and Lasso at alpha=100."""

    def __init__(self):
        self.compare_btn = styled_button('Compare Ridge vs Lasso', width='220px')
        self.compare_btn.on_click(self._on_compare)
        self.output = widgets.Output()

    def _on_compare(self, _):
        alpha = 100

        ridge = Ridge(alpha=alpha)
        lasso = Lasso(alpha=alpha, max_iter=5000)

        ridge.fit(X_train_scaled, y_train)
        lasso.fit(X_train_scaled, y_train)

        # Metrics
        ridge_train_r2 = ridge.score(X_train_scaled, y_train)
        ridge_test_r2 = ridge.score(X_test_scaled, y_test)
        ridge_cv = cross_val_score(Ridge(alpha=alpha), X_train_scaled, y_train, cv=5, scoring='r2')
        ridge_nonzero = int(np.sum(np.abs(ridge.coef_) > 1e-6))

        lasso_train_r2 = lasso.score(X_train_scaled, y_train)
        lasso_test_r2 = lasso.score(X_test_scaled, y_test)
        lasso_cv = cross_val_score(Lasso(alpha=alpha, max_iter=5000), X_train_scaled, y_train, cv=5, scoring='r2')
        lasso_nonzero = int(np.sum(np.abs(lasso.coef_) > 1e-6))

        # Top-20 coefficients for each
        ridge_coefs = pd.Series(ridge.coef_, index=X.columns)
        lasso_coefs = pd.Series(lasso.coef_, index=X.columns)

        ridge_top20 = ridge_coefs.abs().nlargest(20)
        ridge_top20_vals = ridge_coefs[ridge_top20.index].sort_values()

        lasso_top20 = lasso_coefs.abs().nlargest(20)
        lasso_top20_vals = lasso_coefs[lasso_top20.index].sort_values()

        with self.output:
            clear_output(wait=True)

            # Bar charts
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
            apply_dark_theme(fig, [ax1, ax2])

            # Ridge
            r_features = ridge_top20_vals.index.tolist()
            r_values = ridge_top20_vals.values
            r_colors = role_bar_colors(r_features, role_map)
            ax1.barh(range(len(r_features)), r_values, color=r_colors, height=0.7)
            ax1.set_yticks(range(len(r_features)))
            ax1.set_yticklabels(r_features, fontsize=8)
            ax1.set_xlabel('Coefficient Value')
            ax1.set_title(f'Ridge (α=100) — Top 20 Coefficients')
            ax1.axvline(0, color=MUTED_TEXT, linewidth=0.8, alpha=0.5)
            for i, v in enumerate(r_values):
                offset = max(abs(r_values)) * 0.02
                ha = 'left' if v >= 0 else 'right'
                x_pos = v + offset if v >= 0 else v - offset
                ax1.text(x_pos, i, f'{v:.0f}', va='center', ha=ha,
                         color=TEXT_COLOR, fontsize=7)

            # Lasso
            l_features = lasso_top20_vals.index.tolist()
            l_values = lasso_top20_vals.values
            l_colors = role_bar_colors(l_features, role_map)
            ax2.barh(range(len(l_features)), l_values, color=l_colors, height=0.7)
            ax2.set_yticks(range(len(l_features)))
            ax2.set_yticklabels(l_features, fontsize=8)
            ax2.set_xlabel('Coefficient Value')
            ax2.set_title(f'Lasso (α=100) — Top 20 Coefficients')
            ax2.axvline(0, color=MUTED_TEXT, linewidth=0.8, alpha=0.5)
            for i, v in enumerate(l_values):
                offset = max(abs(l_values)) * 0.02 if max(abs(l_values)) > 0 else 1
                ha = 'left' if v >= 0 else 'right'
                x_pos = v + offset if v >= 0 else v - offset
                ax2.text(x_pos, i, f'{v:.0f}', va='center', ha=ha,
                         color=TEXT_COLOR, fontsize=7)

            plt.tight_layout()
            plt.show()

            display(role_legend_html())

            # Performance table
            table_html = f"""
            <div style='margin-top: 15px;'>
            <table style='border-collapse: collapse; width: 700px; background: {AXES_BG};
                          border-radius: 10px; overflow: hidden;'>
                <thead>
                    <tr style='background: #1e293b;'>
                        <th style='padding: 12px 16px; color: {TEXT_COLOR}; text-align: left;
                                   border-bottom: 2px solid #475569;'>Metric</th>
                        <th style='padding: 12px 16px; color: {GREEN}; text-align: center;
                                   border-bottom: 2px solid #475569;'>Ridge (α=100)</th>
                        <th style='padding: 12px 16px; color: #a78bfa; text-align: center;
                                   border-bottom: 2px solid #475569;'>Lasso (α=100)</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td style='padding: 10px 16px; color: {MUTED_TEXT}; border-bottom: 1px solid #334155;'>Train R²</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;
                                   border-bottom: 1px solid #334155;'>{ridge_train_r2:.3f}</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;
                                   border-bottom: 1px solid #334155;'>{lasso_train_r2:.3f}</td>
                    </tr>
                    <tr>
                        <td style='padding: 10px 16px; color: {MUTED_TEXT}; border-bottom: 1px solid #334155;'>Test R²</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;
                                   border-bottom: 1px solid #334155;'>{ridge_test_r2:.3f}</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;
                                   border-bottom: 1px solid #334155;'>{lasso_test_r2:.3f}</td>
                    </tr>
                    <tr>
                        <td style='padding: 10px 16px; color: {MUTED_TEXT}; border-bottom: 1px solid #334155;'>CV R² (5-fold)</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;
                                   border-bottom: 1px solid #334155;'>{ridge_cv.mean():.3f} ± {ridge_cv.std():.3f}</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;
                                   border-bottom: 1px solid #334155;'>{lasso_cv.mean():.3f} ± {lasso_cv.std():.3f}</td>
                    </tr>
                    <tr>
                        <td style='padding: 10px 16px; color: {MUTED_TEXT};'>Non-zero Features</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;'>{ridge_nonzero}</td>
                        <td style='padding: 10px 16px; color: {TEXT_COLOR}; text-align: center;'>{lasso_nonzero}</td>
                    </tr>
                </tbody>
            </table>
            </div>
            """
            display(widgets.HTML(table_html))

            # Summary
            summary = widgets.HTML(f"""
            <div style='background: linear-gradient(135deg, #1e3a5f, #1a1a2e);
                        padding: 18px 22px; border-radius: 10px;
                        border-left: 4px solid {GREEN};
                        margin-top: 15px; max-width: 700px;'>
                <p style='color: {TEXT_COLOR}; font-size: 14px; line-height: 1.6; margin: 0;'>
                    Regularization brought Test R² from -8 to ~0.5 — a massive improvement!
                    But can we do even better?
                </p>
            </div>
            """)
            display(summary)

    def create_ui(self):
        return widgets.VBox([
            self.compare_btn,
            self.output
        ])

comparison_widget = RidgeLassoComparisonWidget()

In [ ]:
#@title ▶️ Launch Ridge vs Lasso Comparison { display-mode: "form" }
display(comparison_widget.create_ui())

## Takeaway

In [ ]:
#@title ▶️ Launch Takeaway { display-mode: "form" }
display(takeaway_box(
    "Regularization penalizes large coefficients. Ridge shrinks all, Lasso zeros out many. "
    "Alpha controls the strength — too low = overfit, too high = underfit. "
    "We went from R²=-8 to R²=0.5. But look at which features Lasso selected..."
))